# SAeUron style steering — smoke test (additive, Eq. 7)

Injects an SAE **style feature direction** into Stable Diffusion so a target style (Van Gogh)
appears **without naming it in the prompt**. Built from the cloned SAeUron repo
(`external/SAeUron`) and the paper (arXiv:2501.18052 — Appendix H / Eq. 7, Sections 4 & 5.1).

**Does not modify** `TestSD.ipynb` or `example_intervention.ipynb`. It reuses the denoising
*conventions* from `TestSD.ipynb` (cpu `Generator` seed, `steps=30`, `guidance=7.5`, matplotlib
display + `savefig`), routed through SAeUron's `HookedStableDiffusionPipeline` so the forward hook
fires inside the UNet.

**How to run:** select the **`diffusion231`** kernel and run top-to-bottom. Section order:
Part 0 (imports) → Part 1 (additive hook) → Part 2 (load) →
**Part 5 wiring smoke test (run first — fast)** → Part 3 (find Van Gogh feature) →
Part 4 (steer + strength sweep).


In [ ]:
# --- setup: ensure SAeUron's hard deps are present in this kernel ---
import sys, subprocess, importlib
for pkg, mod in [("einops", "einops"), ("natsort", "natsort"),
                 ("simple_parsing", "simple_parsing"), ("fire", "fire")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("SAeUron dependencies present")

%matplotlib inline

## Part 0 — Fix the imports

SAeUron uses bare top-level imports (`from SAE...`, `import utils.hooks`). To use it from this repo we
put the clone on `sys.path`. The path is resolved **absolutely** (we search up from the notebook's
working dir for `external/SAeUron`) so it works regardless of where the kernel was launched.

**`utils/` collision check:** if *this* repo had its own top-level `utils/` package, prepending
SAeUron's path would shadow it. This repo has **no** top-level `utils/`, so we take **Branch A**
(prepend to `sys.path`, import `utils.hooks` directly). Branch B (append + load `hooks.py` by file
location under a unique module name) is kept for portability if a `utils/` is ever added here.


In [ ]:
import os, sys, importlib.util

def find_saeuron_path():
    """Search upward from the cwd for external/SAeUron (absolute, cwd-independent)."""
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        cand = os.path.join(d, "external", "SAeUron")
        if os.path.isdir(os.path.join(cand, "SAE")):
            return os.path.abspath(cand)
        d = os.path.dirname(d)
    raise FileNotFoundError("Could not locate external/SAeUron; set SAEURON_PATH manually.")

SAEURON_PATH = find_saeuron_path()
REPO_ROOT = os.path.dirname(os.path.dirname(SAEURON_PATH))   # parent of external/
OUTPUT_DIR = os.path.join(REPO_ROOT, "output_img")           # all figures are saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("SAEURON_PATH =", SAEURON_PATH)
print("OUTPUT_DIR   =", OUTPUT_DIR)

# Does THIS repo (not SAeUron) ship its own top-level utils/ package?
HAS_REPO_UTILS = os.path.isdir(os.path.join(REPO_ROOT, "utils"))
print("this repo has its own top-level utils/ :", HAS_REPO_UTILS)

from importlib import import_module
if not HAS_REPO_UTILS:
    # BRANCH A — no collision: prepend so SAeUron's bare imports resolve, import utils.hooks directly.
    if SAEURON_PATH not in sys.path:
        sys.path.insert(0, SAEURON_PATH)
    from SAE.sae import Sae
    from SAE.hooked_sd_noised_pipeline import HookedStableDiffusionPipeline
    from SAE.unlearning_utils import compute_feature_importance
    import utils.hooks as sae_hooks
    print("Took BRANCH A (prepend sys.path); imported SAeUron's utils.hooks directly.")
else:
    # BRANCH B — collision: append (don't shadow) and load hooks.py by file path under a unique name.
    if SAEURON_PATH not in sys.path:
        sys.path.append(SAEURON_PATH)
    from SAE.sae import Sae
    from SAE.hooked_sd_noised_pipeline import HookedStableDiffusionPipeline
    from SAE.unlearning_utils import compute_feature_importance
    _spec = importlib.util.spec_from_file_location(
        "saeuron_hooks", os.path.join(SAEURON_PATH, "utils", "hooks.py"))
    sae_hooks = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(sae_hooks)
    print("Took BRANCH B (append + file-load hooks.py as 'saeuron_hooks').")

print("SAeUron imports OK. Shipped (multiplicative) hook available as:",
      "sae_hooks.SAEFeatureInterventionHook")

## Part 1 — Additive steering (the key design decision)

The shipped `utils/hooks.py::SAEFeatureInterventionHook` is **multiplicative**:
`latents[..., feature_idx] *= multiplier`. Because the SAE uses **TopK**, a feature that does not
already fire on the prompt has activation `0`, and `0 * multiplier == 0`. So the shipped hook can only
**amplify / ablate a style already present** — it cannot *inject* an absent style. Wrong tool for us.

Instead we implement **additive steering** (paper Appendix H, Eq. 7):

$$F_t \leftarrow F_t + \sum_{i \in F_c} \gamma^{+}_{c}\,\mu(i,t,D_c)\,d_i$$

where $d_i$ is the feature's decoder direction = the *i*-th **column of the SAE decoder** = **row `i` of
`W_dec`** (`W_dec` has shape `[num_latents, d_in]`, and `decode` computes `acts @ W_dec`). We add this
direction into the cross-attention feature map **unconditionally**, whether or not the feature fires.

**Layout check (against `utils/hooks.py`):** the block output is `[2B, C, H, W]` under classifier-free
guidance (first half unconditional, second half conditional). Every shipped hook does
`permute(0,2,3,1).reshape(B, H*W, C)` to feed the SAE — i.e. the **channel axis = axis 1 = `d_in`**.
So we broadcast the direction onto axis 1 with `.view(1, -1, 1, 1)` and add it to the whole
`[2B, C, H, W]` tensor. We assert `W_dec.shape[1] == C` at runtime.


In [ ]:
import torch

class SAEStyleSteeringHook:
    """Additive steering (SAeUron Appendix H, Eq. 7): inject decoder direction(s) into the
    cross-attention feature map, regardless of whether they fire.

    output[0] is [2B, C, H, W] under classifier-free guidance (uncond | cond). We add the (weighted)
    sum of decoder directions to ALL of it, broadcasting on the channel axis (= d_in = W_dec.shape[1]).

    weights: optional per-feature scalars. Set to the feature's avg activation mu(i) to be faithful to
    Eq. 7 (then `strength` plays the role of gamma+). Default = ones (folds gamma+*mu into `strength`).
    """
    def __init__(self, sae, feature_idxs, strength, weights=None):
        self.sae = sae
        self.feature_idxs = torch.as_tensor(feature_idxs, device=sae.W_dec.device).long()
        self.strength = float(strength)
        if weights is None:
            self.weights = torch.ones(len(self.feature_idxs), device=sae.W_dec.device)
        else:
            self.weights = torch.as_tensor(weights, dtype=torch.float32, device=sae.W_dec.device)

    @torch.no_grad()
    def __call__(self, module, input, output):
        hidden = output[0]                                       # [2B, C, H, W]
        dirs = self.sae.W_dec[self.feature_idxs].float()         # [F, C]
        direction = (self.weights.view(-1, 1) * dirs).sum(0)     # [C]
        assert direction.shape[0] == hidden.shape[1], \
            f"W_dec d_in {direction.shape[0]} != channels {hidden.shape[1]}"
        hidden = hidden + self.strength * direction.view(1, -1, 1, 1).to(hidden.dtype)
        return (hidden,)


# --- synthetic shape test: proves the broadcast/sum, needs no model and no download ---
def _synthetic_hook_test():
    class _Fake: pass
    f = _Fake(); C = 8
    f.W_dec = torch.zeros(20, C); f.W_dec[3, 5] = 1.0            # feature 3 -> channel 5
    h = SAEStyleSteeringHook(f, [3], strength=2.0)
    x = torch.zeros(2, C, 4, 4)                                 # [2B, C, H, W]
    out = h(None, None, (x,))[0]
    assert out.shape == x.shape
    assert torch.allclose(out[:, 5], torch.full((2, 4, 4), 2.0))   # channel 5 shifted by strength
    assert out[:, [c for c in range(C) if c != 5]].abs().sum() == 0  # others untouched
    print("synthetic additive-hook test passed; output shape", tuple(out.shape))

_synthetic_hook_test()

## Part 2 — Model + SAE pairing

- **Base model:** `CompVis/stable-diffusion-v1-4`, loaded through `HookedStableDiffusionPipeline`.
- **Style hookpoint:** `unet.up_blocks.1.attentions.2` ("up.1.2") — the style block (paper §5.1;
  objects use up.1.1).
- **Style SAE:** `bcywinski/SAeUron` at that hookpoint.

**Device / dtype:** CUDA → `float16`; otherwise MPS / CPU → **`float32`** (fp16 on MPS/CPU is unreliable
for SD). This machine has no CUDA, so it runs on MPS/fp32 — slower, but the identical code runs on a
single GPU in fp16.

> ⚠️ **Model-mismatch caveat.** The `bcywinski/SAeUron` style SAE was trained on activations from the
> **UnlearnCanvas fine-tuned SD checkpoint** (an SD-v1.5 fine-tune), *not* vanilla SD-1.4. For pure
> **additive** steering (Eq. 7) this is tolerable — we add a decoder *direction*; we do **not** rely on
> the SAE to faithfully reconstruct vanilla-SD activations — so it's fine for a quick smoke test. If the
> effect is weak, the principled fix is to load the UnlearnCanvas SD checkpoint (linked in the SAeUron
> README) instead of 1.4. We do **not** download it here.


In [ ]:
import torch

if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float32      # fp16 on MPS is unreliable for SD
else:
    device, dtype = "cpu", torch.float32
print("device:", device, "| dtype:", dtype)

MODEL_NAME       = "CompVis/stable-diffusion-v1-4"
STYLE_HOOKPOINT  = "unet.up_blocks.1.attentions.2"   # up.1.2 -> style  (paper §5.1)
OBJECT_HOOKPOINT = "unet.up_blocks.1.attentions.1"   # up.1.1 -> objects

pipe = HookedStableDiffusionPipeline.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, safety_checker=None,
).to(device)
pipe.set_progress_bar_config(disable=True)
print("loaded", MODEL_NAME, "(scheduler forced to DDIM by HookedStableDiffusionPipeline)")

In [ ]:
style_sae = Sae.load_from_hub("bcywinski/SAeUron", hookpoint=STYLE_HOOKPOINT, device=device).to(dtype)

print("style SAE  W_dec:", tuple(style_sae.W_dec.shape),
      "| d_in:", style_sae.d_in, "| num_latents:", style_sae.num_latents)
print("           cfg: k =", style_sae.cfg.k,
      "| input_unit_norm =", style_sae.cfg.input_unit_norm,
      "| normalize_decoder =", style_sae.cfg.normalize_decoder)
# W_dec is [num_latents, d_in]; row i is the steering direction d_i for feature i.

## Part 5 — Wiring smoke test (run this FIRST)

Confirms the hook plumbing independently of feature-finding, using the **known-good pairing** from
`example_intervention.ipynb`: vanilla SD-1.4 + the `bcywinski/SAeUron_coco` SAE on
`unet.up_blocks.1.attentions.1`, additively steering an arbitrary feature (`11627`). If the steered
image visibly differs from the baseline, the additive hook is wired correctly. (On MPS this is much
faster than the Part 3 scoring, which is why it comes first.)


In [ ]:
import matplotlib.pyplot as plt

def show_row(images, titles, suptitle=None, save=None):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=10); ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=110, bbox_inches="tight"); print("saved", save)
    plt.show()

def generate(prompt, hook=None, hookpoint=None, seed=40, steps=30, guidance=7.5):
    """Reuses TestSD's conventions (cpu Generator seed / 30 steps / guidance 7.5),
    routed through run_with_hooks so the forward hook fires inside the UNet."""
    gen = torch.Generator(device="cpu").manual_seed(seed)
    hooks = {hookpoint: hook} if hook is not None else {}
    imgs = pipe.run_with_hooks(
        prompt=prompt, generator=gen,
        num_inference_steps=steps, guidance_scale=guidance,
        device=torch.device(device),
        position_hook_dict=hooks,
    )
    return imgs[0]

# known-good COCO SAE on the object block (matches example_intervention.ipynb)
coco_sae = Sae.load_from_hub("bcywinski/SAeUron_coco", hookpoint=OBJECT_HOOKPOINT, device=device).to(dtype)
print("coco SAE W_dec:", tuple(coco_sae.W_dec.shape), "| num_latents:", coco_sae.num_latents)

SMOKE_FEATURE = 11627
assert SMOKE_FEATURE < coco_sae.num_latents, \
    f"pick a feature < {coco_sae.num_latents}"

smoke_prompt = "a photo of a cat"
base_smoke    = generate(smoke_prompt, hook=None)
smoke_hook    = SAEStyleSteeringHook(coco_sae, [SMOKE_FEATURE], strength=8.0)
steered_smoke = generate(smoke_prompt, hook=smoke_hook, hookpoint=OBJECT_HOOKPOINT)

show_row([base_smoke, steered_smoke],
         ["baseline (no hook)", f"+additive feat {SMOKE_FEATURE} (strength=8)"],
         suptitle="Part 5: wiring smoke test (up.1.1, COCO SAE)",
         save=os.path.join(OUTPUT_DIR, "smoke_test.png"))
# If the two images look identical, bump strength (e.g. 16, 32) -- additive scale depends on activation magnitude.

## Part 3 — Find the Van Gogh style feature

No precomputed index ships with the repo, so we derive it cheaply (no UnlearnCanvas data, no full
`gather_sae_acts_ca_prompts.py` run). We replicate that script's convention **exactly**:
`run_with_cache(..., output_type="latent")` caches the up.1.2 outputs `[num_prompts, steps, H*W, d_in]`
(the cache keeps only the text-conditioned half, per §5.1); then per prompt `sae.encode` → scatter
top-k into `[steps*H*W, num_latents]` → reshape → **mean over patches** → `[steps, num_latents]`,
stacked → `[num_prompts, steps, num_latents]` — exactly the layout
`compute_feature_importance(style_latents_dict, target_style, timestep)` expects.

**Cheap-smoke-test simplifications (documented):** 8 short content prompts; `SCORING_STEPS = 20`
(instead of 50); 4 style sets — **Van Gogh** (target) vs Cubism / Watercolor / Pop Art. We average the
importance score over the cached timesteps and take the **argmax** (paper uses τ=1 for styles → one
feature). On MPS this is the slow part; lower the counts/steps to iterate faster.


In [ ]:
CONTENT_PROMPTS = ["a house", "a cat", "a mountain", "a bicycle",
                   "a tree", "a boat", "a chair", "a bridge"]
STYLE_SETS = {
    "Van_Gogh":   "in Van Gogh style",
    "Cubism":     "in Cubism style",
    "Watercolor": "in Watercolor style",
    "Pop_Art":    "in Pop Art style",
}
SCORING_STEPS    = 20      # reduced from 50 for a cheap smoke test (documented)
SCORING_GUIDANCE = 9.0     # matches gather_sae_acts_ca_prompts.py
SCORING_SEED     = 188

def encode_style_latents(acts, sae, steps):
    """[num_prompts, steps, H*W, d_in] cached acts -> [num_prompts, steps, num_latents]
    (replicates gather_sae_acts_ca_prompts.py: per-prompt encode, scatter top-k, mean over patches)."""
    out = []
    with torch.no_grad():
        for i in range(acts.shape[0]):
            sae_in = acts[i].reshape(steps, -1, sae.d_in).to(sae.device).to(sae.dtype)
            top_acts, top_indices = sae.encode(sae_in)                      # PatchTopK after load
            buf = torch.zeros((top_acts.shape[0], sae.num_latents),
                              device=sae.device, dtype=top_acts.dtype)
            scattered = buf.scatter(-1, top_indices, top_acts)
            scattered = scattered.reshape(steps, -1, sae.num_latents)       # [steps, H*W, num_latents]
            out.append(scattered.mean(1).float().cpu())                     # mean over patches
    return torch.stack(out)                                                 # [num_prompts, steps, num_latents]

def free_mem():
    if device == "mps":
        torch.mps.empty_cache()

# IMPORTANT: cache one prompt at a time. Batching many prompts (x2 for CFG) OOMs MPS
# in the 512x512 self-attention. On a big CUDA GPU you can batch the whole list at once.
style_latents_dict = {}
for name, suffix in STYLE_SETS.items():
    per_prompt = []
    for c in CONTENT_PROMPTS:
        _, cache = pipe.run_with_cache(
            prompt=[f"{c} {suffix}"],
            generator=torch.Generator(device="cpu").manual_seed(SCORING_SEED),
            num_inference_steps=SCORING_STEPS,
            guidance_scale=SCORING_GUIDANCE,
            positions_to_cache=[STYLE_HOOKPOINT],
            save_output=True,
            output_type="latent",                 # skip VAE decode -- we only need activations
            device=torch.device(device),
        )
        acts = cache["output"][STYLE_HOOKPOINT]    # [1, steps, H*W, d_in]  (cpu)
        assert acts.shape[-1] == style_sae.d_in, (acts.shape, style_sae.d_in)   # channel == d_in
        per_prompt.append(encode_style_latents(acts, style_sae, SCORING_STEPS))
        del cache, acts; free_mem()
    style_latents_dict[name] = torch.cat(per_prompt, dim=0)   # [num_prompts, steps, num_latents]
    print(f"{name:11s} -> latents {tuple(style_latents_dict[name].shape)}")

In [ ]:
# average the score over cached timesteps, then argmax (tau=1 for styles -> a single feature)
scores = torch.stack([
    compute_feature_importance(style_latents_dict, "Van_Gogh", t)
    for t in range(SCORING_STEPS)
]).mean(0)

vangogh_idx = int(scores.argmax())
topv, topi = scores.topk(5)
print("Van Gogh feature index:", vangogh_idx)
print("top-5 (feature_idx, score):",
      list(zip(topi.tolist(), [round(v, 4) for v in topv.tolist()])))

# avg activation mu of that feature on Van Gogh samples (for faithful Eq. 7 scaling below)
mu_vangogh = style_latents_dict["Van_Gogh"][:, :, vangogh_idx].mean().item()
print("mu (avg activation of Van Gogh feature):", round(mu_vangogh, 4))

## Part 4 — Steer and generate (reusing TestSD conventions)

We reuse `TestSD.ipynb`'s denoising conventions — `torch.Generator(device="cpu").manual_seed(...)`,
`steps=30`, `guidance=7.5`, matplotlib display + `savefig` — but route through `pipe.run_with_hooks(...)`
so the additive forward hook fires inside the UNet at up.1.2. The prompt (`"a photo of a cat"`)
**never mentions Van Gogh**. We show baseline vs steered, then a strength sweep `[0, 2, 4, 8, 16]`
(strength 0 == baseline, same seed throughout so only the steering varies).


In [ ]:
NEUTRAL_PROMPT = "a photo of a cat"     # never mentions Van Gogh
STEER_SEED     = 40

base_vg    = generate(NEUTRAL_PROMPT, hook=None, seed=STEER_SEED)
vg_hook    = SAEStyleSteeringHook(style_sae, [vangogh_idx], strength=8.0)
steered_vg = generate(NEUTRAL_PROMPT, hook=vg_hook, hookpoint=STYLE_HOOKPOINT, seed=STEER_SEED)

show_row([base_vg, steered_vg],
         ["baseline", f"+Van Gogh feat {vangogh_idx} (strength=8)"],
         suptitle=f"Part 4: style injection on a neutral prompt — '{NEUTRAL_PROMPT}'",
         save=os.path.join(OUTPUT_DIR, "vangogh_steer.png"))

In [ ]:
STRENGTHS = [0, 2, 4, 8, 16]
imgs, titles = [], []
for s in STRENGTHS:
    hook = SAEStyleSteeringHook(style_sae, [vangogh_idx], strength=float(s)) if s > 0 else None
    imgs.append(generate(NEUTRAL_PROMPT, hook=hook, hookpoint=STYLE_HOOKPOINT, seed=STEER_SEED))
    titles.append(f"strength={s}")

show_row(imgs, titles,
         suptitle=f"Van Gogh feature {vangogh_idx} — strength sweep on '{NEUTRAL_PROMPT}'",
         save=os.path.join(OUTPUT_DIR, "vangogh_sweep.png"))

In [ ]:
# Optional: faithful Eq. 7 scaling -- strength = gamma+ * mu  (mu from Part 3).
gamma_plus = 4.0
faithful_hook = SAEStyleSteeringHook(style_sae, [vangogh_idx], strength=gamma_plus * mu_vangogh)
faithful_img  = generate(NEUTRAL_PROMPT, hook=faithful_hook, hookpoint=STYLE_HOOKPOINT, seed=STEER_SEED)

show_row([base_vg, faithful_img],
         ["baseline", f"Eq.7: gamma+={gamma_plus}, mu={mu_vangogh:.3f}"],
         suptitle="Part 4 (faithful Eq. 7 mu-scaling)")

## Notes & follow-ups

- **Multiplicative vs additive:** the shipped hook multiplies a TopK latent, so it cannot inject an
  *absent* style (`0 * m = 0`); additive steering (Eq. 7) adds the decoder direction unconditionally.
- **Faithful Eq. 7:** set `strength = γ⁺ · μ` (the last cell). The raw-strength sweep folds `γ⁺·μ` into
  one scalar, which is fine for a smoke test.
- **Setting difference:** the paper's Appendix H steers **unconditional** (empty-prompt) generations;
  here we inject into a neutral **content** prompt — a slightly harder setting.
- **If the effect is weak:** (1) increase strength; (2) raise `SCORING_STEPS` / prompt count for a
  sharper feature index; (3) the principled fix — load the **UnlearnCanvas SD** checkpoint (SD-v1.5
  fine-tune, linked in the SAeUron README) instead of vanilla 1.4, since the SAE was trained on its
  activations.
- **Performance:** runs on MPS in fp32 (slow) or a single CUDA GPU in fp16. Lower steps/counts to iterate.
